In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

In [2]:
#อ่านไฟล์จาก test_set.csv

df = pd.read_csv("D:/New Finetune Hackathon/Train Stage 1/All Data stage 1.csv")
df.head(5)

,query log,status,label
0,SELECT * FROM users WHERE username = 'admin' O...,anomaly,0
1,SELECT id FROM accounts WHERE email = '' UNION...,anomaly,0
2,SELECT * FROM products WHERE id = 10; DROP TAB...,anomaly,0
3,SELECT * FROM orders WHERE order_id = 105 OR 1...,anomaly,0
4,SELECT * FROM customers WHERE name = 'a'/**/OR...,anomaly,0


In [4]:
#โหลด Model
path = "D:/New Finetune Hackathon/Train Stage 3/Finetuned Bert Model State 3/checkpoint-273 (best)"
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path)


# ตรวจสอบว่ามี GPU ให้ใช้งานหรือไม่ ถ้ามีให้ใช้ cuda ถ้าไม่มีให้ใช้ cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#บังคับว่าต้อง inference ที่ GPU (ถ้ามี)
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [5]:
#1 ทำ preprocessing สำหรับ log (For Model Stage 2)

def add_prefix_token(text): # log data ต้องผ่าน code นี้ก่อน training / inference
    # clean log
    text = text.replace("\t", " ")
    text = text.strip()
    # add token
    if text[0].isalpha() or text[3].isalpha():
        return "[SQL]\n" + text
    else:
        return "[LOG]\n" + text

In [ ]:
def predict_log(log_text):
    log_text = add_prefix_token(log_text)
    inputs = tokenizer(
        log_text,
        return_tensors="pt",
        truncation=True,
        padding=True, # ใส่เผื่อเอาไว้ตอน inference มากกว่า 1 log (Batch Size > 1)
        max_length=128
    )

    # ✅ ย้าย inputs ไป device เดียวกับ model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]

    #return "NORMAL" if pred == 1 else "ANOMALY" ,prob
    #return "normaly" if pred == 1 else "anomaly" ,prob # All Data Stage 1
    return "Normal" if pred == 1 else "anormaly" ,prob

In [7]:
#วัด Accuracy

correct_predictions = 0
total_predictions = len(df)

for index,row in df.iterrows():
    text_to_classify = row['query log'] # ใช้คอลัมน์ 'query log' เป็น input
    true_label = row['status'] # ใช้คอลัมน์ 'status' เป็นคำตอบจริง (normal หรือ anomaly)

    # ทำนาย
    prediction_result,confidence = predict_log(text_to_classify)

    # ตรวจสอบว่าทำนายถูกต้องหรือไม่
    if prediction_result == true_label:
        correct_predictions += 1
        correction = 'True'
    else:
        correction = 'False'
    print(f"prediction = {prediction_result} | true_status = {true_label} | correction = {correction} | confidence = {confidence}")

prediction = anormaly | true_status = anomaly | correction = False | confidence = [0.9878694415092468, 0.012130588293075562]
prediction = anormaly | true_status = anomaly | correction = False | confidence = [0.986316978931427, 0.013683028519153595]
prediction = anormaly | true_status = anomaly | correction = False | confidence = [0.9547781348228455, 0.04522183537483215]
prediction = anormaly | true_status = anomaly | correction = False | confidence = [0.9797842502593994, 0.02021576091647148]
prediction = anormaly | true_status = anomaly | correction = False | confidence = [0.9834873080253601, 0.016512660309672356]
prediction = anormaly | true_status = anomaly | correction = False | confidence = [0.9863030314445496, 0.013696981593966484]
prediction = anormaly | true_status = anomaly | correction = False | confidence = [0.9818586111068726, 0.018141377717256546]
prediction = anormaly | true_status = anomaly | correction = False | confidence = [0.9860166907310486, 0.013983349315822124]
pre

In [ ]:
# วัด Accuracy
accuracy = (correct_predictions / total_predictions) * 100
print(f"test_set จำนวน {total_predictions}")
print(f"ทำนายถูกจำนวน {correct_predictions}")
print(f"Accuracy = {accuracy}")